# Day 9 — Gradient boosting: fitting to the residuals

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss

## Setup — reuse Day 6/7/8's Titanic cleaning

Same 4 features as Day 6/7/8 (`pclass`, `fare`, `who`, `family_size`), same split. Like Day 8, we take raw numpy views (`X_train`, `X_test`, `y_train_v`, `y_test_v`) — the boosting loop below does additive updates to a raw score array `F`, which needs plain array math, not a DataFrame.

In [ ]:
df = sns.load_dataset("titanic")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"C": 0, "Q": 1, "S": 2})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df.drop(
    columns=[
        "class",
        "embark_town",
        "alive",
        "alone",
        "sibsp",
        "parch",
        "deck",
        "adult_male",
    ],
    inplace=True,
)

median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
mode_embarked = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)
df["embarked"] = df["embarked"].astype(int)
df["who"] = df["who"].map({"man": 0, "woman": 1, "child": 2})
df.drop(["sex", "age", "embarked"], axis=1, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)
X_train, X_test = x_train.values.astype(float), x_test.values.astype(float)
y_train_v, y_test_v = y_train.values.astype(float), y_test.values.astype(float)

print("train shape:", X_train.shape, " test shape:", X_test.shape)

## Step 1 — what boosting actually is, before any code

**The core idea.** An ensemble that builds models *sequentially*, where each new model's entire job is to correct the errors the ensemble so far still gets wrong. Contrast directly with Day 8's random forest, still fresh: bagging trains many trees **independently and in parallel** on bootstrap resamples, then averages their votes — each tree is a full, unrelated attempt at the whole problem, and averaging cancels out their individual variance. Boosting trains trees **one at a time, in sequence**, and each tree only ever looks at what's still wrong after all the previous trees' contributions — there is no averaging-away of independent noise, because the trees aren't independent by construction.

**What "weak learner" means here.** Day 7/8's trees were tuned for standalone accuracy (`max_depth=3` chosen by CV as the *best single tree*). Boosting deliberately uses much shallower trees (`max_depth=2`, sometimes even depth-1 "stumps") — a single one is a bad, weak model on purpose. It only needs to be slightly better than random at predicting *the current residual*; it doesn't need to solve the classification problem alone, because dozens or hundreds of these weak correctors get summed together.

**Bias vs. variance, the one-line version.** Bagging (Day 8) primarily fights **variance** — averaging many high-variance, low-bias trees smooths out their individual noise. Boosting primarily fights **bias** — a single shallow, "weak" tree underfits badly on its own, and boosting's whole mechanism is adding more of them, each targeted at the specific error the ensemble still has, until that bias is driven down.

**Why "fit a tree to the residual" is the whole trick.** If you already have a model that outputs a probability `p` for each row and you know the true label `y`, then `y - p` tells you, per row, how wrong and in which direction the current model is. A regression tree that predicts `y - p` well is a tree that has learned a correction — add a small, learning-rate-scaled slice of that correction to the running score, and the ensemble's predictions move toward the truth without needing to relearn what previous trees already got right.

**Why log-odds, not probability, is what actually gets summed.** Predicted probabilities are bounded in `[0, 1]`, so they can't be added indefinitely without breaking — clip a probability to `[0,1]` and adding more terms saturates immediately. The **raw score** `F` (the log-odds / logit, same quantity Day 6's `z = Xw + b` produced before the sigmoid) has no such bound, which is exactly why the update is `F += lr * tree.predict(X)` on the unbounded score, with `sigmoid(F)` only applied at the moment a probability is actually needed (for the residual, or for a final prediction).

## Step 2 — the Day 6 parallel, made literal

Day 6's logistic-regression update was `w -= lr * dw` with `dw = (1/n) X^T (p - y)`. Gradient boosting's update is `F += lr * tree.predict(X)`, where the tree is fit to approximate `y - p`. Same residual driving both — "gradient descent in function space," not a loose analogy: Day 6 moved a *coefficient vector* in the direction that reduces error; boosting fits a *small tree* that approximates that same direction as a function of the input, then adds it in. Both are, mechanically, "take a step against the gradient of the loss," just parameterized differently (a linear weight vector vs. an arbitrary piecewise-constant function).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def gradient_boost_fit(X, y, n_rounds, lr=0.1, max_depth=2, seed=42):
    p0 = y.mean()
    F0 = np.log(p0 / (1 - p0))  # log-odds init
    F = np.full(len(y), F0)
    trees = []
    for i in range(n_rounds):
        p = sigmoid(F)
        residual = y - p  # same pseudo-residual as Day 6's error term
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=seed + i)
        tree.fit(X, residual)
        F += lr * tree.predict(X)
        trees.append(tree)
    return F0, trees


def gradient_boost_predict_F(F0, trees, lr, X):
    F = np.full(X.shape[0], F0)
    for tree in trees:
        F += lr * tree.predict(X)
    return F

## Step 3 — round sweep

Sweep `n_rounds` at a fixed `lr=0.1, max_depth=2`, tracking train/test accuracy **and** train/test log-loss at each round count.

In [ ]:
for n_rounds in [1, 5, 10, 25, 50, 100, 200]:
    F0, trees = gradient_boost_fit(
        X_train, y_train_v, n_rounds=n_rounds, lr=0.1, max_depth=2
    )

    F_train = gradient_boost_predict_F(F0, trees, 0.1, X_train)
    F_test = gradient_boost_predict_F(F0, trees, 0.1, X_test)
    p_train, p_test = sigmoid(F_train), sigmoid(F_test)

    train_acc = accuracy_score(y_train_v, (p_train >= 0.5).astype(int))
    test_acc = accuracy_score(y_test_v, (p_test >= 0.5).astype(int))
    train_ll = log_loss(y_train_v, p_train)
    test_ll = log_loss(y_test_v, p_test)

    print(
        f"rounds={n_rounds:>3}: train acc={train_acc:.4f}, test acc={test_acc:.4f}, "
        f"train ll={train_ll:.4f}, test ll={test_ll:.4f}"
    )

Two real patterns in the actual numbers, not one. Train accuracy climbs steadily through round 50 (0.6236 → 0.8343) and keeps inching up after that (0.8371, 0.8385) — each additional round genuinely chips away at remaining training error, the defining boosting behavior. Test accuracy climbs in step through round 50 (reaching 0.8156) and then goes completely flat — rounds 100 and 200 both land at exactly 0.8156, the classic sign that extra rounds are now fitting train-only noise rather than anything that generalizes.

Test log-loss, though, keeps dropping through round 200 (0.4700 → 0.4387 → 0.4269) even while test *accuracy* is frozen. The model's probability estimates keep getting better-calibrated well after its hard 0.5-threshold decisions stop changing — the same distinction Day 1's `.describe()` vs. raw stats work first raised: a metric built on a threshold (accuracy) and a metric built on the full probability (log-loss) don't have to move together, and here they visibly don't.

## Step 4 — learning rate: a confound worth naming

In [ ]:
for lr_try in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]:
    F0, trees = gradient_boost_fit(
        X_train, y_train_v, n_rounds=100, lr=lr_try, max_depth=2
    )
    F_test = gradient_boost_predict_F(F0, trees, lr_try, X_test)
    test_acc = accuracy_score(y_test_v, (sigmoid(F_test) >= 0.5).astype(int))
    print(f"lr={lr_try:<5} test acc={test_acc:.4f}")

Tempting to read this as "bigger `lr` wins" — accuracy is non-decreasing all the way from `lr=0.01` (0.7709) to `lr=1.0` (0.8324). Don't draw that conclusion yet: this comparison is confounded. At a fixed 100-round budget, a bigger `lr` just makes more progress per round, so the small-`lr` runs may simply not have been given enough rounds to catch up — not that they're worse in principle. `lr` and `n_rounds` are a matched pair that need joint tuning, not two independently-optimizable knobs — which is exactly what Step 5's CV grid search does properly.

## Step 5 — verify against sklearn, then tune it for real

In [ ]:
sk_gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42
)
sk_gb.fit(x_train, y_train)
sk_test_acc = accuracy_score(y_test, sk_gb.predict(x_test))

F0, trees = gradient_boost_fit(X_train, y_train_v, n_rounds=100, lr=0.1, max_depth=2)
F_test = gradient_boost_predict_F(F0, trees, 0.1, X_test)
scratch_test_acc = accuracy_score(y_test_v, (sigmoid(F_test) >= 0.5).astype(int))

print(
    f"sklearn GradientBoostingClassifier (n=100, lr=0.1, depth=2): test acc={sk_test_acc:.4f}"
)
print(
    f"From-scratch, same settings:                                  test acc={scratch_test_acc:.4f}"
)

In [ ]:
n_estimators_grid = [10, 50, 100, 200]
lr_grid = [0.01, 0.05, 0.1, 0.2, 0.3]

results = []
for n_est in n_estimators_grid:
    for lr_try in lr_grid:
        gb = GradientBoostingClassifier(
            n_estimators=n_est, learning_rate=lr_try, max_depth=2, random_state=42
        )
        scores = cross_val_score(gb, x_train, y_train, cv=5)
        results.append((n_est, lr_try, scores.mean(), scores.std()))

results.sort(key=lambda r: -r[2])
print("top 5 configs by CV mean:")
for n_est, lr_try, mean, std in results[:5]:
    print(
        f"  n_estimators={n_est:>3}, lr={lr_try:<4}: CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n, best_lr, best_cv, best_std = results[0]
print(
    f"\nBest: n_estimators={best_n}, lr={best_lr}, CV acc={best_cv:.4f} (+/-{best_std:.4f})"
)

best_gb = GradientBoostingClassifier(
    n_estimators=best_n, learning_rate=best_lr, max_depth=2, random_state=42
)
best_gb.fit(x_train, y_train)
test_preds = best_gb.predict(x_test)
final_test_acc = accuracy_score(y_test, test_preds)
cm = confusion_matrix(y_test, test_preds)

print(f"\nFinal ONE-TIME test accuracy: {final_test_acc:.4f}")
print("confusion matrix:\n", cm)

| Model | CV accuracy | Test accuracy |
|---|---|---|
| Day 6 logistic regression (L2) | — | 0.8101 |
| Day 7 single tree (CV-tuned) | 0.8286 | 0.8212 |
| Day 8 random forest (CV-tuned) | 0.8370 | 0.8212 |
| Day 9 gradient boosting (CV-tuned) | **0.8384** | 0.8212 |

Gradient boosting edges out the highest CV score of the four models, but lands on the *exact same* test accuracy as Day 7's tree and Day 8's forest — three different algorithms converging on 0.8212290502793296 to 13 decimal places. Worth checking whether that's the same 147 correct calls or a coincidence: it isn't the same calls. Day 7/8's confusion matrix was `[[92,13],[19,55]]`; Day 9's is `[[91,14],[18,56]]` — one fewer true negative, one more true positive, net zero. Same accuracy, different mistakes.

Don't read a 0.0014-0.0184 CV gap as a real winner on 712 training rows — the honest read across all three ensembles is "these are performing similarly on this dataset and this feature set," not "boosting won." What *is* real: gradient boosting reached the same test ceiling as random forest using its own, structurally different mechanism (sequential bias-reduction vs. parallel variance-reduction) — that convergence is itself the useful result, not a horse race.

## Step 7 — feature importances

In [ ]:
print("Gradient boosting feature importances:")
for name, imp in sorted(
    zip(x_train.columns, best_gb.feature_importances_), key=lambda t: -t[1]
):
    print(f"  {name:<12}: {imp:.4f}")

print("\nDay 7 single tree (max_depth=3) importances, for comparison:")
day7 = {"who": 0.6277, "pclass": 0.2191, "fare": 0.1532, "family_size": 0.0000}
for name, imp in day7.items():
    print(f"  {name:<12}: {imp:.4f}")

print(
    "\nDay 8 random forest (n_estimators=50, max_depth=8) importances, for comparison:"
)
day8 = {"who": 0.4244, "fare": 0.3382, "pclass": 0.1199, "family_size": 0.1175}
for name, imp in day8.items():
    print(f"  {name:<12}: {imp:.4f}")